<a href="https://colab.research.google.com/github/BekenAidana/AI_agents/blob/main/AgentWithTools_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'OPENAI_API_KEY' as a secret over there with the "key" icon
#
#
# You will also want to add some sample files to the "Files" (folder icon)
# on the left. When the agent asks you what to do, start with something
# simplie like "tell me what files are in this directory"
#
import os
from google.colab import userdata
api_key = userdata.get('GEMINI_API_KEY')
os.environ['GEMINI_API_KEY'] = api_key

In [2]:
import json
import os
import sys
from litellm import completion
from typing import List, Dict

def extract_markdown_block(response: str, block_type: str = "json") -> str:
    """Extract code block from response"""

    if not '```' in response:
        return response

    code_block = response.split('```')[1].strip()

    if code_block.startswith(block_type):
        code_block = code_block[len(block_type):].strip()

    return code_block

def generate_response(messages: List[Dict]) -> str:
    """Call LLM to get a response."""
    response = completion(
        model="gemini/gemini-2.5-flash",
        messages=messages,
        max_tokens=1024
    )
    return response.choices[0].message.content.strip()

def parse_action(response: str) -> Dict:
    """Parse the LLM response into a structured action dictionary."""
    try:
        response = extract_markdown_block(response, "action")
        response_json = json.loads(response)
        if "tool_name" in response_json and "args" in response_json:
            return response_json
        else:
            return {"tool_name": "error", "args": {"message": "You must respond with a JSON tool invocation."}}
    except json.JSONDecodeError:
        return {"tool_name": "error", "args": {"message": "Invalid JSON response. You must respond with a JSON tool invocation."}}

def list_files() -> List[str]:
    """List files in the current directory."""
    return os.listdir(".")

def read_file(file_name: str) -> str:
    """Read a file's contents."""
    try:
        with open(file_name, "r") as file:
            return file.read()
    except FileNotFoundError:
        return f"Error: {file_name} not found."
    except Exception as e:
        return f"Error: {str(e)}"

# Define system instructions (Agent Rules)
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

Available tools:

```json
{
    "list_files": {
        "description": "Lists all files in the current directory.",
        "parameters": {}
    },
    "read_file": {
        "description": "Reads the content of a file.",
        "parameters": {
            "file_name": {
                "type": "string",
                "description": "The name of the file to read."
            }
        }
    },
    "terminate": {
        "description": "Ends the agent loop and provides a summary of the task.",
        "parameters": {
            "message": {
                "type": "string",
                "description": "Summary message to return to the user."
            }
        }
    }
}
```

If a user asks about files, documents, or content, first list the files before reading them.

When you are done, terminate the conversation by using the "terminate" tool and I will provide the results to the user.

Important!!! Every response MUST have an action.
You must ALWAYS respond in this format:

<Stop and think step by step. Parameters map to args. Insert a rich description of your step by step thoughts here.>

```action
{
    "tool_name": "insert tool_name",
    "args": {...fill in any required arguments here...}
}
```"""
}]


In [3]:
# Initialize agent parameters
def new_chat():
  iterations = 0
  max_iterations = 10

  user_task = input("What would you like me to do? ")

  memory = [{"role": "user", "content": user_task}]

  # The Agent Loop
  while iterations < max_iterations:
      # 1. Construct prompt: Combine agent rules with memory
      prompt = agent_rules + memory

      # 2. Generate response from LLM
      print("Agent thinking...")
      response = generate_response(prompt)
      print(f"Agent response: {response}")

      # 3. Parse response to determine action
      action = parse_action(response)
      result = "Action executed"

      if action["tool_name"] == "list_files":
          result = {"result": list_files()}
      elif action["tool_name"] == "read_file":
          result = {"result": read_file(action["args"]["file_name"])}
      elif action["tool_name"] == "error":
          result = {"error": action["args"]["message"]}
      elif action["tool_name"] == "terminate":
          print(action["args"]["message"])
          break
      else:
          result = {"error": "Unknown action: " + action["tool_name"]}

      print(f"Action result: {result}")

      # 5. Update memory with response and results
      memory.extend([
          {"role": "assistant", "content": response},
          {"role": "user", "content": json.dumps(result)}
      ])

      # 6. Check termination condition
      if action["tool_name"] == "terminate":
          break

      iterations += 1

In [4]:
new_chat()

What would you like me to do? Какие файлы есть у меня в директории?
Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent thinking...
Agent response: ```action
{
    "tool_name": "terminate",
    "args": {
        "message": "В вашей директории есть следующие файлы: .config, sample_data."
    }
}
```
В вашей директории есть следующие файлы: .config, sample_data.


In [5]:
new_chat()

What would you like me to do? можешь ли ты сказать, что находится в файле sample_data
Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent thinking...
Agent response: ```action
{
    "tool_name": "read_file",
    "args": {
        "file_name": "sample_data"
    }
}
```
Action result: {'result': "Error: [Errno 21] Is a directory: 'sample_data'"}
Agent thinking...
Agent response: ```action
{
    "tool_name": "terminate",
    "args": {
        "message": "Файл 'sample_data' является директорией, а не файлом, поэтому я не могу прочитать его содержимое как обычный файл."
    }
}
```
Файл 'sample_data' является директорией, а не файлом, поэтому я не могу прочитать его содержимое как обычный файл.


In [7]:
new_chat()

What would you like me to do? можешь ли сказать, что находится в папке sample_data
Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent thinking...
Agent response: Я не могу напрямую заглянуть в папку "sample_data" с помощью доступных мне инструментов, так как `list_files` не поддерживает указание пути к папке. Он перечисляет только файлы в текущем каталоге.

```action
{
    "tool_name": "terminate",
    "args": {
        "message": "Извините, я не могу напрямую просмотреть содержимое папки 'sample_data', так как мои инструменты не поддерживают указание пути для 'list_files'. Я могу только перечислить файлы в текущем каталоге."
    }
}
```
Извините, я не могу напрямую просмотреть содержимое папки 'sample_data', так как мои инструменты не поддерживают указание пути для 'list_files'. Я могу только перечислить файлы в текущем каталоге.


In [9]:
new_chat()

What would you like me to do? Что за файл config?
Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent thinking...
Agent response: ```action
{
    "tool_name": "read_file",
    "args": {
        "file_name": ".config"
    }
}
```
Action result: {'result': "Error: [Errno 21] Is a directory: '.config'"}
Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {
        "file_name": ".config"
    }
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent thinking...
Agent response: ```action
{
    "tool_name": "terminate",
    "args": {
        "message": "Файл под названием `config` не является файлом, это директория (.config)."
    }
}
```
Файл под названием `config` не является файлом, это директория (.config).


In [10]:
new_chat()

What would you like me to do? Не могу найти файл Айдана
Agent thinking...
Agent response: ```action
{
    "tool_name": "list_files",
    "args": {}
}
```
Action result: {'result': ['.config', 'sample_data']}
Agent thinking...
Agent response: ```action
{
    "tool_name": "terminate",
    "args": {
        "message": "Файл с именем \"Айдана\" не был найден в текущем каталоге. Доступные файлы: .config, sample_data."
    }
}
```
Файл с именем "Айдана" не был найден в текущем каталоге. Доступные файлы: .config, sample_data.


In [11]:
new_chat()

What would you like me to do? Приготовь пиццу
Agent thinking...
Agent response: Я не могу приготовить пиццу, так как являюсь языковой моделью и не обладаю физическими возможностями для приготовления еды. Мои задачи заключаются в обработке информации и использовании предоставленных инструментов.
```action
{
    "tool_name": "terminate",
    "args": {
        "message": "Извините, я не могу приготовить пиццу, так как я языковая модель и не обладаю физическими возможностями."
    }
}
```
Извините, я не могу приготовить пиццу, так как я языковая модель и не обладаю физическими возможностями.


In [12]:
new_chat()

What would you like me to do? Напиши рецепт как приготовить пиццу
Agent thinking...
Agent response: Я не могу написать рецепт, так как у меня нет такой функциональности. Мои возможности ограничены только файловыми операциями.

```action
{
    "tool_name": "terminate",
    "args": {
        "message": "К сожалению, я не могу написать рецепт, так как у меня нет такой функции. Мои возможности ограничены только файловыми операциями."
    }
}
```
К сожалению, я не могу написать рецепт, так как у меня нет такой функции. Мои возможности ограничены только файловыми операциями.
